In [97]:
from bs4 import BeautifulSoup
import asyncio, sys
from concurrent.futures import ThreadPoolExecutor
from markdownify import markdownify as md
from urllib.parse import urljoin
import json
def _crawl(url):
    """Run Playwright in a separate thread to avoid Jupyter event loop conflict."""
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url, wait_until="domcontentloaded", timeout=60000)
        html = page.content()
        browser.close()
    return html

# url = "https://www.kapruka.com/buyonline/signature-men-s-formal-slim-fi/kid/ef_pc_clot0v2248pod01092p"
# url = "https://www.kapruka.com/buyonline/mallika-hemachandra-22kt-gold-/kid/jewellerymh091"
# url = "https://www.kapruka.com/buyonline/scarlet-petal-vanilla-sponge-g/kid/cake00ka002112"






In [ ]:
def extract_main_content(soup: BeautifulSoup, url: str):
    details_div = soup.find("div", id='Tab1')

    title = soup.title.string if soup.title else url.split("/")[-1]

    test = details_div.find_all(["p", "ul"])
    content = ""
    for i in test:
        if "<p>" in str(i):
            p_content = i.get_text()
            content += p_content + "\n\n"
        elif "<ul>" in str(i):
            features = "\n".join([li.get_text(strip=True) for li in details_div.find_all('li')])
            content += features + "\n\n"
    return content
    
def extract_meta(soup, prop):
    """ Return content of a <meta property="..."> tag. """
    tag = soup.find("meta", property=prop)
    return tag['content'].strip() if tag and tag.get("content") else ""

def extract_sku(soup):
    el = soup.find("input",{"id": "id", "type": "hidden"})
    if el and el.get("value"):
        return el["value"]
    return extract_meta(soup, "product:retailer_item_id")


def extract_links(soup, url: str):
    links = []
    for a in soup.find_all("a", href=True):
        href = a.get("href", '')
        if not href:
            continue

        if href.startswith('/'):
            href = "https://www.kapruka.com" + href
        elif not href.startswith('http'):
            href = urljoin(url, href)

        if href.startswith( "https://www.kapruka.com"):
            href = href.split('#')[0].split('?')[0]
            if href and href !=url:
                links.append(href)
    return links


def extract_varients(soup: BeautifulSoup, url: str):
    script_tags = soup.find_all('script')
    products_data = {}
    toppers = []
    links = extract_links(soup, url)
    print(links)
    for script in script_tags:
        if script.string and 'let products' in script.string:
            match = re.search(r'let products\s*=\s*(\[.*?\]);', script.string, re.DOTALL) 
            if match:
                products_data = json.loads(match.group(1))
                break 
        else:
            availability = extract_meta(soup,"product:availability")
            price_lkr = extract_meta(soup, "product:price:amount")
            product_sku = extract_sku(soup)

            all_products = []
            product_map = []
            text = script.string or ""
            if "allProducts" in text:
                match_toppers = re.search(
                    r'var\s+allProducts\s*=\s*(\[.*?\]);',
                    text
                )
                if match_toppers:
                    all_products = json.loads(match_toppers.group(1))

            if "productMapJson" in text:
                match_price = re.search(r'var\s+productMapJson\s*=\s*(\{.*?\});', text, re.DOTALL)
                if match_price:
                    try:
                        product_map = json.loads(match_price.group(1))
                    except json.JSONDecodeError:
                        pass
            if len(toppers) == 0:
                if all_products and product_map:

                    for product in all_products:
                        pid = product.get("productID", "")
                        topper = {
                            "productID": pid,
                            "name": product.get("name", ""),
                            "price_lkr": product_map.get(pid, "N/A"),
                            "price_usd": product.get("price", ""),
                            "available": product.get("available", ""),
                            "type": product.get("type", ""),
                            "imageName": product.get("imageName", ""),
                            "deliveryType": product.get("deliveryType", ""),
                            "bestseller": product.get("bestseller", ""),
                        }
                        toppers.append(topper)

        price_valid_until = ""
        try:
            data = json.loads(script.string or "")
            if str(data.get("@type", "")).lower() in ("product",):
                offers = data.get("offers", [{}])
                if isinstance(offers, list):
                    offers = offers[0]
                    price_lkr         = price_lkr or str(offers.get("price", ""))
                    price_valid_until  = offers.get("priceValidUntil", "")
                    avail_url          = offers.get("availability", "")
                    available          = "InStock" in avail_url or available
        except Exception:
            pass

        products_data = {
            "availabilty" :"in stock" if availability else "Out of stock",
            "price_lkr" : price_lkr, 
            "product sku" : product_sku,
            "price valid until" : price_valid_until,
            "toppers" : toppers
        }
                
            

    return products_data

In [99]:
urls = [
    "https://www.kapruka.com/buyonline/signature-men-s-formal-slim-fi/kid/ef_pc_clot0v2248pod01092p",
    "https://www.kapruka.com/buyonline/mallika-hemachandra-22kt-gold-/kid/jewellerymh091",
    "https://www.kapruka.com/buyonline/scarlet-petal-vanilla-sponge-g/kid/cake00ka002112"
]

for url in urls:
    with ThreadPoolExecutor(max_workers=1) as executor:
        html = executor.submit(_crawl, url).result()

    soup = BeautifulSoup(html, "html.parser")
    title = soup.title
    main_content = extract_main_content(soup, url)
    others = extract_varients(soup, url)

    print(title)
    print(main_content)
    print(others)

['https://www.kapruka.com/online/newyear', 'https://www.kapruka.com/online/easter', 'https://www.kapruka.com/online/newyear', 'https://www.kapruka.com/online/easter', 'https://www.kapruka.com/online/cakes', 'https://www.kapruka.com/online/cakes/price/kapruka_cakes', 'https://www.kapruka.com/online/cakes/price/java', 'https://www.kapruka.com/online/cakes/price/galadari', 'https://www.kapruka.com/online/cakes/price/nh_collection', 'https://www.kapruka.com/online/cakes/price/breadtalk', 'https://www.kapruka.com/online/cakes/price/colombo_hilton', 'https://www.kapruka.com/online/cakes/price/kingsbury', 'https://www.kapruka.com/online/cakes/price/cinnamon_lakeside', 'https://www.kapruka.com/online/cakes/price/gerard_mendis_chocolatier', 'https://www.kapruka.com/online/cakes/price/waters_edge', 'https://www.kapruka.com/online/cakes/price/divine', 'https://www.kapruka.com/online/cakes/price/shangri-la', 'https://www.kapruka.com/online/cakes/price/mahaweli_reach_kandy', 'https://www.kapruka.co

Looking for a cake delivered fast? Kapruka offers fresh cakes with guaranteed delivery in Sri Lanka.

Indulge in the Scarlet Petal Vanilla Sponge Glaze New Year Cake, a decadent delight perfect for celebrating special occasions in Sri Lanka. This exquisite cake combines a luscious vanilla sponge infused with gateaux syrup and smothered in smooth whipping cream, making it an irresistible treat for any festivity.

Crafted with a vanilla sponge base for a soft, moist texture.
Soaked in rich gateaux syrup, enhancing every bite.
Topped with alily-shaped whitechocolateflowerfor a touch of elegance.
Finished with a radiant red glaze, adding visual allure.
Accented with shimmering gold paper, perfect for New Yearâ``scelebrations.
Weighs 2.11 lbs (1.0 kg), ideal for sharing with loved ones.
Suitable for occasions like New Year, birthdays, and anniversaries in Sri Lanka.

Bring home the Scarlet Petal Vanilla Sponge Glaze New Year Cake today from Kapruka and elevate your celebration.




{'availa

In [88]:
print(md(main_content))

Looking for a cake delivered fast? Kapruka offers fresh cakes with guaranteed delivery in Sri Lanka.
Indulge in the Scarlet Petal Vanilla Sponge Glaze New Year Cake, a decadent delight perfect for celebrating special occasions in Sri Lanka. This exquisite cake combines a luscious vanilla sponge infused with gateaux syrup and smothered in smooth whipping cream, making it an irresistible treat for any festivity.
Crafted with a vanilla sponge base for a soft, moist texture.
Soaked in rich gateaux syrup, enhancing every bite.
Topped with alily-shaped whitechocolateflowerfor a touch of elegance.
Finished with a radiant red glaze, adding visual allure.
Accented with shimmering gold paper, perfect for New Yearâ``scelebrations.
Weighs 2.11 lbs (1.0 kg), ideal for sharing with loved ones.
Suitable for occasions like New Year, birthdays, and anniversaries in Sri Lanka.
Bring home the Scarlet Petal Vanilla Sponge Glaze New Year Cake today from Kapruka and elevate your celebration.
